**Reconhecimento de Padrões**

Grupo:

*   João
*   Marcus
* Priscilla

**USP - SIN5007--2026**



# Importação de Dados e Pré-Processamento Parcial

Etapas de Pré-processamento

1) **Data Cleaning**
*   Replace ? por Nan
*   Drop Nan & Duplicates
*   Inconsistências - 'label noise ' - identifical features for different targets
*   Retirada de fnlwgt (baixa correl vs target, qdo comparada a outras features)


2) **Transformação Log**


*   tratamento de variáveis numéricas com alto grau de assimetria

3) **Tratamento de Outliers**

*  Retirada de Outliers via método de Tukey (retirada de valores extremos +/- 1.5 * IQR

4) Feature Engineering
5) Discretização
6) Codificação de variáveis categóricas



Dataset escolhido: https://archive.ics.uci.edu/dataset/2/adult

##Import Data

In [ ]:
#instalacao necessaria para ler repositorio

!pip install ucimlrepo imbalanced-learn
#!pip install mixed-naive-bayes

#essa biblioteca em geral converte valores estranhos em NaN (exemplo: ?)




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder
from scipy.stats import chi2_contingency, pointbiserialr
from imblearn.under_sampling import RandomUnderSampler
import heapq

#from mixed_naive_bayes import MixedNB

from sklearn.datasets import make_classification
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
# CÓDIGO ORIGINAL FORNECIDO:


# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "adult.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "wenruliu/adult-income-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)


print("First 5 records:", df.head())

Using Colab cache for faster access to the 'adult-income-dataset' dataset.
First 5 records:    age  workclass  fnlwgt     education  educational-num      marital-status  \
0   25    Private  226802          11th                7       Never-married   
1   38    Private   89814       HS-grad                9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm               12  Married-civ-spouse   
3   44    Private  160323  Some-college               10  Married-civ-spouse   
4   18          ?  103497  Some-college               10       Never-married   

          occupation relationship   race  gender  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                  ?    Own

In [ ]:
X = df.drop('income', axis=1)
y = df[['income']]

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')

Shape de X: (48842, 14)
Shape de y: (48842, 1)


## Limpeza: Ruídos, Inconsistências e Alvo

### Subtask:
Standardize the target variable, replace placeholders with NaN, and eliminate duplicates and label noise (conflicting records).


In [ ]:

# 1. Standardize the 'income' column (target)
df_cleaned = pd.concat([X, y], axis=1)
target_col = y.columns[0]

df_cleaned[target_col] = df_cleaned[target_col].astype(str).str.replace('.', '', regex=False).str.strip()

# 2. Replace the character '?' with np.nan
df_cleaned = df_cleaned.replace('?', np.nan)

# 3. Drop rows with NaN and remove exact duplicates
initial_count = len(df_cleaned)
df_cleaned = df_cleaned.dropna()
df_cleaned = df_cleaned.drop_duplicates()

# 4. Identify and remove 'label noise' (identical features with different targets)
feature_cols = list(X.columns)
inconsistent_mask = df_cleaned.groupby(feature_cols)[target_col].transform('nunique') > 1
df_cleaned = df_cleaned[~inconsistent_mask]

# 6. Drop de fnlwgt
df_cleaned = df_cleaned.drop(columns=['fnlwgt'])

# 5. Print results
final_count = len(df_cleaned)
print(f'Initial row count: {initial_count}')
print(f'Final row count after cleaning and noise removal: {final_count}')
print(f'Total rows removed: {initial_count - final_count}')

display(df_cleaned.head())

Initial row count: 48842
Final row count after cleaning and noise removal: 45165
Total rows removed: 3677


,age,workclass,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
5,34,Private,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K


## Transformação de Escala (Log)

### Subtask:
Aplicar transformação logarítmica (np.log1p) nas variáveis com alta assimetria (fnlwgt, capital-gain, capital-loss) para aproximá-las de uma distribuição normal.


In [ ]:

# 1. Select the columns with high skewness identified in EDA
columns_to_log = ['capital-gain', 'capital-loss']

# 2. Check skewness before transformation for reference
skew_before = df_cleaned[columns_to_log].skew()
print('--- Skewness Before Log Transformation ---')
print(skew_before)

# 3. Apply np.log1p (log(1+x)) to the selected columns
# This handles 0 values gracefully and reduces skewness
df_cleaned[columns_to_log] = np.log1p(df_cleaned[columns_to_log])

# 4. Check skewness after transformation to confirm normalization progress
skew_after = df_cleaned[columns_to_log].skew()
print('\n--- Skewness After Log Transformation ---')
print(skew_after)

# 5. Display the first few rows to confirm the change in values
display(df_cleaned[columns_to_log].head())

--- Skewness Before Log Transformation ---
capital-gain    11.781500
capital-loss     4.513051
dtype: float64

--- Skewness After Log Transformation ---
capital-gain    3.079867
capital-loss    4.268074
dtype: float64


,capital-gain,capital-loss
0,0.000000,0.0
1,0.000000,0.0
2,0.000000,0.0
3,8.947546,0.0
5,0.000000,0.0


## Tratamento de Outliers

### Subtask:
Avaliar e tratar outliers nas variáveis contínuas, utilizando técnicas de clipping (limitação de valores) para reduzir o impacto de valores extremos sem perder informação.


In [ ]:

# 1. Identify continuous numerical variables
# We focus on the log-transformed versions of capital-gain/loss and the original hours-per-week
continuous_vars = ['capital-gain', 'capital-loss', 'hours-per-week']

# Store stats before clipping
stats_before = df_cleaned[continuous_vars].describe().T

print('--- Applying Clipping (Tukey Method) ---')

for col in continuous_vars:
    # 2. Calculate IQR
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1

    # Define limits
    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    # 3. Apply clipping
    df_cleaned[col] = df_cleaned[col].clip(lower=lower_limit, upper=upper_limit)
    print(f'Feature: {col} | Limits: [{lower_limit:.2f}, {upper_limit:.2f}]')

# 4. Verification
stats_after = df_cleaned[continuous_vars].describe().T

print('\n--- Descriptive Statistics Comparison ---')
comparison = pd.concat([stats_before[['mean', 'std', 'min', 'max']],
                        stats_after[['mean', 'std', 'min', 'max']]],
                       axis=1, keys=['Before Clipping', 'After Clipping'])

display(comparison)

--- Applying Clipping (Tukey Method) ---
Feature: capital-gain | Limits: [0.00, 0.00]
Feature: capital-loss | Limits: [0.00, 0.00]
Feature: hours-per-week | Limits: [32.50, 52.50]

--- Descriptive Statistics Comparison ---


Before Clipping                            After Clipping  \
                          mean        std  min        max           mean   
capital-gain          0.741694   2.467943  0.0  11.512925       0.000000   
capital-loss          0.355937   1.596870  0.0   8.379539       0.000000   
hours-per-week       40.942278  12.008902  1.0  99.000000      41.419229   

                                      
                     std   min   max  
capital-gain    0.000000   0.0   0.0  
capital-loss    0.000000   0.0   0.0  
hours-per-week  6.146139  32.5  52.5

## Engenharia de Características

### Subtask:
Criar novas features, como 'capital-balance' (gain - loss), e agrupar categorias de baixa frequência ou alta similaridade em colunas como 'education' e 'marital-status'.


In [ ]:



# 2. Define education mapping dictionary
edu_map = {
    'Preschool': 'dropout',
    '1st-4th': 'dropout',
    '5th-6th': 'dropout',
    '7th-8th': 'dropout',
    '9th': 'dropout',
    '10th': 'dropout',
    '11th': 'dropout',
    '12th': 'dropout',
    'HS-grad': 'HighGrad',
    'Some-college': 'CommunityCollege',
    'Assoc-acdm': 'CommunityCollege',
    'Assoc-voc': 'CommunityCollege',
    'Bachelors': 'Bachelors',
    'Masters': 'Masters',
    'Prof-school': 'Masters',
    'Doctorate': 'Doctorate'
}

# 3. Apply education mapping
df_cleaned['education'] = df_cleaned['education'].replace(edu_map)

# 4. Define marital-status mapping dictionary
marital_map = {
    'Never-married': 'NotMarried',
    'Married-civ-spouse': 'Married',
    'Married-AF-spouse': 'Married',
    'Married-spouse-absent': 'NotMarried',
    'Divorced': 'Separated',
    'Separated': 'Separated',
    'Widowed': 'Widowed'
}

# 5. Apply marital-status mapping and display results
df_cleaned['marital-status'] = df_cleaned['marital-status'].replace(marital_map)



# Drop education-num
df_cleaned = df_cleaned.drop(columns=['educational-num'])

# Label encoding para sex
df_cleaned['gender'] = pd.Categorical(df_cleaned['gender']).codes


# 1. Feature: capital-balance
if 'capital-gain' in df_cleaned.columns and 'capital-loss' in df_cleaned.columns:
    df_cleaned['capital-balance'] = df_cleaned['capital-gain'] - df_cleaned['capital-loss']
if 'capital-balance' in df_cleaned.columns:
    df_cleaned = df_cleaned.drop(columns=['capital-gain', 'capital-loss'])

# 7. Agrupar paises: manter apenas os 3 mais frequentes e o resto em 'Outros'
if 'native-country' in df_cleaned.columns:
    top3 = df_cleaned['native-country'].value_counts().head(3).index
    df_cleaned['native-country'] = df_cleaned['native-country'].where(df_cleaned['native-country'].isin(top3), 'Outros')


print('--- New Education Categories ---')
print(df_cleaned['education'].value_counts())
print('\n--- New Marital Status Categories ---')
print(df_cleaned['marital-status'].value_counts())

display(df_cleaned.head())

--- New Education Categories ---
education
HighGrad            14764
CommunityCollege    13352
Bachelors            7557
dropout              5652
Masters              3296
Doctorate             544
Name: count, dtype: int64

--- New Marital Status Categories ---
marital-status
Married       21066
NotMarried    15117
Separated      7705
Widowed        1277
Name: count, dtype: int64


,age,workclass,education,marital-status,occupation,relationship,race,gender,hours-per-week,native-country,income,capital-balance
0,25,Private,dropout,NotMarried,Machine-op-inspct,Own-child,Black,1,40.0,United-States,<=50K,0.0
1,38,Private,HighGrad,Married,Farming-fishing,Husband,White,1,50.0,United-States,<=50K,0.0
2,28,Local-gov,CommunityCollege,Married,Protective-serv,Husband,White,1,40.0,United-States,>50K,0.0
3,44,Private,CommunityCollege,Married,Machine-op-inspct,Husband,Black,1,40.0,United-States,>50K,0.0
5,34,Private,dropout,NotMarried,Other-service,Not-in-family,White,1,32.5,United-States,<=50K,0.0


## Discretização

### Subtask:
Aplicar técnicas de discretização em colunas como 'age' e 'hours-per-week' para capturar padrões não lineares no DataFrame `df_cleaned`.


In [ ]:

# 1. Discretize 'age' into 5 quantiles
df_cleaned['age_bin'] = pd.qcut(df_cleaned['age'], q=5)

# 2. Discretize 'hours-per-week' into custom logical bins
# Bins: 0-35 (Part-time), 35-45 (Full-time), 45-100 (Overtime)
bins = [0, 35, 45, 100]
labels = ['Part-time', 'Full-time', 'Overtime']
df_cleaned['hours_bin'] = pd.cut(df_cleaned['hours-per-week'], bins=bins, labels=labels)

# 3. Verify distributions
print('--- Distribution of age_bin ---')
print(df_cleaned['age_bin'].value_counts().sort_index())

print('\n--- Distribution of hours_bin ---')
print(df_cleaned['hours_bin'].value_counts().sort_index())

# 4. Remove original numerical columns to avoid redundancy
df_cleaned = df_cleaned.drop(columns=['age', 'hours-per-week'])

print('\nOriginal columns removed. df_cleaned shape:', df_cleaned.shape)
display(df_cleaned[['age_bin', 'hours_bin']].head())

--- Distribution of age_bin ---
age_bin
(16.999, 26.0]    9508
(26.0, 34.0]      9702
(34.0, 41.0]      8472
(41.0, 50.0]      8803
(50.0, 90.0]      8680
Name: count, dtype: int64

--- Distribution of hours_bin ---
hours_bin
Part-time     8788
Full-time    26112
Overtime     10265
Name: count, dtype: int64

Original columns removed. df_cleaned shape: (45165, 12)


,age_bin,hours_bin
0,"(16.999, 26.0]",Full-time
1,"(34.0, 41.0]",Overtime
2,"(26.0, 34.0]",Full-time
3,"(41.0, 50.0]",Full-time
5,"(26.0, 34.0]",Part-time


## Codificação de Variáveis Categóricas

### Subtask:
Aplicar One-Hot Encoding nas variáveis nominais e Ordinal Encoding naquelas que possuem hierarquia, transformando o dataset em formato puramente numérico.


In [ ]:

# 1. Define the hierarchical order for 'education'
edu_order = ['dropout', 'HighGrad', 'CommunityCollege', 'Bachelors', 'Masters', 'Doctorate']

# 2. Apply Ordinal Encoding to 'education'
# The 'education' column has already been numerically mapped in a previous step
# based on the kernel state, so this step is now redundant and causing errors.
# df_cleaned['education'] = OrdinalEncoder(categories=[edu_order]).fit_transform(df_cleaned[['education']])

# 3. Convert target 'income' to binary numeric (0 and 1) in a robust way
# Ensure income values are clean strings before mapping
df_cleaned['income'] = df_cleaned['income'].astype(str).str.strip()
# Use np.where for robust conversion to 0 or 1, converting unmatched to NaN
df_cleaned['income'] = np.where(df_cleaned['income'] == '>50K', 1,
                                np.where(df_cleaned['income'] == '<=50K', 0, np.nan))

# Drop rows where 'income' is NaN after conversion, ensuring target is clean
df_cleaned.dropna(subset=['income'], inplace=True)
df_cleaned['income'] = df_cleaned['income'].astype(int) # Convert to int as 0 or 1

# 4. Dynamically identify nominal variables for One-Hot Encoding
# Find columns that are of 'object' or 'category' dtype and are not 'income'
actual_nominal_cols_to_encode = [col for col in df_cleaned.columns
                                 if (df_cleaned[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df_cleaned[col]))
                                 and col != 'income']

print(f"Categorical columns identified for One-Hot Encoding: {actual_nominal_cols_to_encode}")

# 5. Apply One-Hot Encoding with drop_first=True
if actual_nominal_cols_to_encode: # Only proceed if there are columns to encode
    df_cleaned = pd.get_dummies(df_cleaned, columns=actual_nominal_cols_to_encode, drop_first=True)
else:
    print("No remaining categorical columns to one-hot encode.")

# 6. Verify that all columns are numeric
print('--- Data Types Summary ---')
print(df_cleaned.dtypes.value_counts())

# Final check for non-numeric columns
non_numeric = df_cleaned.select_dtypes(exclude=[np.number, 'bool']).columns
print(f'\nNon-numeric columns remaining: {list(non_numeric)}')

display(df_cleaned.head())

Categorical columns identified for One-Hot Encoding: ['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country', 'age_bin', 'hours_bin']
--- Data Types Summary ---
bool       45
int8        1
int64       1
float64     1
Name: count, dtype: int64

Non-numeric columns remaining: []


/tmp/ipykernel_7587/861255710.py:23: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  if (df_cleaned[col].dtype == 'object' or pd.api.types.is_categorical_dtype(df_cleaned[col]))


,gender,income,capital-balance,workclass_Local-gov,workclass_Private,workclass_Self-emp-inc,workclass_Self-emp-not-inc,workclass_State-gov,workclass_Without-pay,education_CommunityCollege,...,race_White,native-country_Outros,native-country_Philippines,native-country_United-States,"age_bin_(26.0, 34.0]","age_bin_(34.0, 41.0]","age_bin_(41.0, 50.0]","age_bin_(50.0, 90.0]",hours_bin_Full-time,hours_bin_Overtime
0,1,0,0.0,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,True,False
1,1,0,0.0,False,True,False,False,False,False,False,...,True,False,False,True,False,True,False,False,False,True
2,1,1,0.0,True,False,False,False,False,False,True,...,True,False,False,True,True,False,False,False,True,False
3,1,1,0.0,False,True,False,False,False,False,True,...,False,False,False,True,False,False,True,False,True,False
5,1,0,0.0,False,True,False,False,False,False,False,...,True,False,False,True,True,False,False,False,False,False


# Preparação dos Dados para o Treinamento

Primeiro, vamos separar as características (X) do conjunto de dados `df_cleaned` da variável alvo (y), que é 'income'.

In [ ]:
X = df_cleaned.drop('income', axis=1)
y = df_cleaned['income']

print(f'Shape de X: {X.shape}')
print(f'Shape de y: {y.shape}')

Shape de X: (45165, 47)
Shape de y: (45165,)


#Aplicação de Naive Bayes

## Todas as características

### Versão: Dataset Completo (Naive Bayes com StratifiedKFold, Otimizado para F1-Score e k=10)

Esta seção replica o experimento de otimização de F1-score e avaliação multi-métrica para a versão que utiliza o dataset completo, sem seleção de características ou redução de dimensionalidade, apenas escalonamento e o classificador Gaussian Naive Bayes.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score, recall_score, precision_score

# 1. Defina o Pipeline
pipe_full_f1 = Pipeline([
    ('scaler', StandardScaler()),
    ('nb', GaussianNB())
])

# 2. Defina o espaço de busca
param_grid_full_f1 = {
    'nb__var_smoothing': [1e-9, 1e-8, 1e-7] # Parâmetro para Gaussian Naive Bayes
}

# 3. Defina as métricas de scoring para cross_validate
scoring_metrics_full = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score)
}

# 4. Loop Interno: Ajuste fino (Tuning) com a métrica F1 e k=10
inner_cv_full_f1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
clf_full_f1 = GridSearchCV(pipe_full_f1, param_grid=param_grid_full_f1, cv=inner_cv_full_f1, n_jobs=-1, scoring='f1')

# 5. Loop Externo: Avaliação Real (sem vazamento) com múltiplas métricas e k=10
outer_cv_full_f1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
nested_scores_full_f1 = cross_validate(clf_full_f1, X, y, cv=outer_cv_full_f1, n_jobs=-1, scoring=scoring_metrics_full, return_estimator=True)

print(f"--- Versão: Dataset Completo (Naive Bayes com StratifiedKFold) Otimizado para F1-Score ---")

print("\nMétricas da Validação Aninhada (Nested Cross-Validation):")
for metric_name, scores in nested_scores_full_f1.items():
    if metric_name.startswith('test_'): # Filtra apenas as pontuações de teste
        print(f"  {metric_name[5:].replace('_', ' ').capitalize()}: Média = {scores.mean():.2f}, Desvio Padrão = {scores.std():.2f}")

# Para obter o melhor resultado (parâmetros e score) do GridSearchCV após o Nested CV,
# precisamos ajustar o GridSearchCV uma vez no dataset completo.
clf_full_f1.fit(X, y)
print(f"\nMelhor F1-score (GridSearchCV no dataset completo): {clf_full_f1.best_score_:.2f}")
print(f"Melhores parâmetros (GridSearchCV no dataset completo): {clf_full_f1.best_params_}")

# Métricas do modelo final ajustado no dataset completo
y_pred_full = clf_full_f1.predict(X)
print("\nMétricas do Modelo Final Ajustado (no dataset completo):")
print(f"  Acurácia: {accuracy_score(y, y_pred_full):.2f}")
print(f"  F1-Score: {f1_score(y, y_pred_full):.2f}")
print(f"  Recall: {recall_score(y, y_pred_full):.2f}")
print(f"  Precisão: {precision_score(y, y_pred_full):.2f}")

--- Versão: Dataset Completo (Naive Bayes com StratifiedKFold) Otimizado para F1-Score ---

Métricas da Validação Aninhada (Nested Cross-Validation):
  Accuracy: Média = 0.75, Desvio Padrão = 0.01
  F1: Média = 0.62, Desvio Padrão = 0.01
  Recall: Média = 0.84, Desvio Padrão = 0.01
  Precision: Média = 0.49, Desvio Padrão = 0.01

Melhor F1-score (GridSearchCV no dataset completo): 0.62
Melhores parâmetros (GridSearchCV no dataset completo): {'nb__var_smoothing': 1e-09}

Métricas do Modelo Final Ajustado (no dataset completo):
  Acurácia: 0.75
  F1-Score: 0.62
  Recall: 0.84
  Precisão: 0.50


### Versão 5: Dataset Completo com Balanceamento de Classes (Undersampling) Otimizado para F1-Score

Esta versão aplica o `RandomUnderSampler` diretamente no pipeline, sem a etapa de PCA. O objetivo é avaliar o desempenho do Gaussian Naive Bayes no dataset completo, com classes balanceadas no treinamento/validação, otimizando para F1-Score e avaliando múltiplas métricas.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, recall_score, precision_score

# 1. Defina o Pipeline com Undersampling (sem PCA)
pipe_full_undersample = ImbPipeline([
    ('scaler', StandardScaler()),
    ('undersampler', RandomUnderSampler(random_state=42)), # Undersampling para balancear as classes no treino
    ('nb', GaussianNB())
])

# 2. Defina o espaço de busca
param_grid_full_undersample = {
    'nb__var_smoothing': [1e-9, 1e-8, 1e-7]
}

# 3. Defina as métricas de scoring para cross_validate
scoring_metrics_full_undersample = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score)
}

# 4. Loop Interno: Ajuste fino (Tuning) com a métrica F1
inner_cv_full_undersample = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# O GridSearchCV otimiza baseado no F1-score
clf_full_undersample = GridSearchCV(pipe_full_undersample, param_grid=param_grid_full_undersample,
                                    cv=inner_cv_full_undersample, n_jobs=-1, scoring='f1')

# 5. Loop Externo: Avaliação Real (sem vazamento) com múltiplas métricas
outer_cv_full_undersample = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
nested_scores_full_undersample = cross_validate(clf_full_undersample, X, y, cv=outer_cv_full_undersample,
                                                  n_jobs=-1, scoring=scoring_metrics_full_undersample, return_estimator=True)

print(f"--- Versão: Dataset Completo, Undersampling e Naive Bayes Otimizado para F1-Score ---")

print("\nMétricas da Validação Aninhada (Nested Cross-Validation):")
for metric_name, scores in nested_scores_full_undersample.items():
    if metric_name.startswith('test_'): # Filtra apenas as pontuações de teste
        print(f"  {metric_name[5:].replace('_', ' ').capitalize()}: Média = {scores.mean():.2f}, Desvio Padrão = {scores.std():.2f}")

# Para obter o melhor resultado (parâmetros e score) do GridSearchCV após o Nested CV,
# precisamos ajustar o GridSearchCV uma vez no dataset completo.
clf_full_undersample.fit(X, y)
print(f"\nMelhor F1-score (GridSearchCV no dataset completo): {clf_full_undersample.best_score_:.2f}")
print(f"Melhores parâmetros (GridSearchCV no dataset completo): {clf_full_undersample.best_params_}")

# Métricas do modelo final ajustado no dataset completo
y_pred_full_undersample = clf_full_undersample.predict(X)
print("\nMétricas do Modelo Final Ajustado (no dataset completo):")
print(f"  Acurácia: {accuracy_score(y, y_pred_full_undersample):.2f}")
print(f"  F1-Score: {f1_score(y, y_pred_full_undersample):.2f}")
print(f"  Recall: {recall_score(y, y_pred_full_undersample):.2f}")
print(f"  Precisão: {precision_score(y, y_pred_full_undersample):.2f}")

--- Versão: Dataset Completo, Undersampling e Naive Bayes Otimizado para F1-Score ---

Métricas da Validação Aninhada (Nested Cross-Validation):
  Accuracy: Média = 0.73, Desvio Padrão = 0.01
  F1: Média = 0.61, Desvio Padrão = 0.01
  Recall: Média = 0.85, Desvio Padrão = 0.01
  Precision: Média = 0.48, Desvio Padrão = 0.01

Melhor F1-score (GridSearchCV no dataset completo): 0.61
Melhores parâmetros (GridSearchCV no dataset completo): {'nb__var_smoothing': 1e-09}

Métricas do Modelo Final Ajustado (no dataset completo):
  Acurácia: 0.74
  F1-Score: 0.61
  Recall: 0.85
  Precisão: 0.48


## PCA

### Versão: Com Redução de Dimensionalidade (PCA) e Sem Balanceamento de Classes


Esta seção demonstra como especificar uma métrica de pontuação diferente (e.g., 'f1' para o F1-score) no `GridSearchCV` para otimizar os hiperparâmetros com base nessa métrica em vez da acurácia padrão. k=10 e estratificado.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_validate # Import cross_validate
from sklearn.metrics import make_scorer, accuracy_score, f1_score, recall_score, precision_score # Import all required metrics

# 1. Defina o Pipeline (usaremos o mesmo pipeline PCA para consistência)
pipe_pca_f1 = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('nb', GaussianNB())
])

# 2. Defina o espaço de busca
param_grid_pca_f1 = {
    'pca__n_components': [2, 5, 10],
    'nb__var_smoothing': [1e-9, 1e-8, 1e-7]
}

# 3. Defina as métricas de scoring para cross_validate
scoring_metrics = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score)
}

# 4. Loop Interno: Ajuste fino (Tuning) com a métrica F1
inner_cv_pca_f1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) # k=10
# O GridSearchCV ainda otimiza baseado no F1-score
clf_pca_f1 = GridSearchCV(pipe_pca_f1, param_grid=param_grid_pca_f1, cv=inner_cv_pca_f1, n_jobs=-1, scoring='f1')

# 5. Loop Externo: Avaliação Real (sem vazamento) com múltiplas métricas
outer_cv_pca_f1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42) # k=10
nested_scores_pca_f1 = cross_validate(clf_pca_f1, X, y, cv=outer_cv_pca_f1, n_jobs=-1, scoring=scoring_metrics, return_estimator=True)

print(f"--- Versão: Com Redução de Dimensionalidade (PCA e Naive Bayes) Otimizado para F1-Score ---")

print("\nMétricas da Validação Aninhada (Nested Cross-Validation):")
for metric_name, scores in nested_scores_pca_f1.items():
    if metric_name.startswith('test_'): # Filtra apenas as pontuações de teste
        print(f"  {metric_name[5:].replace('_', ' ').capitalize()}: Média = {scores.mean():.2f}, Desvio Padrão = {scores.std():.2f}")

# Para obter o melhor resultado (parâmetros e score) do GridSearchCV após o Nested CV,
# precisamos ajustar o GridSearchCV uma vez no dataset completo.
clf_pca_f1.fit(X, y)
print(f"\nMelhor F1-score (GridSearchCV no dataset completo): {clf_pca_f1.best_score_:.2f}")
print(f"Melhores parâmetros (GridSearchCV no dataset completo): {clf_pca_f1.best_params_}")

# Métricas do modelo final ajustado no dataset completo
y_pred = clf_pca_f1.predict(X)
print("\nMétricas do Modelo Final Ajustado (no dataset completo):")
print(f"  Acurácia: {accuracy_score(y, y_pred):.2f}")
print(f"  F1-Score: {f1_score(y, y_pred):.2f}")
print(f"  Recall: {recall_score(y, y_pred):.2f}")
print(f"  Precisão: {precision_score(y, y_pred):.2f}")

--- Versão: Com Redução de Dimensionalidade (PCA e Naive Bayes) Otimizado para F1-Score ---

Métricas da Validação Aninhada (Nested Cross-Validation):
  Accuracy: Média = 0.82, Desvio Padrão = 0.00
  F1: Média = 0.58, Desvio Padrão = 0.01
  Recall: Média = 0.50, Desvio Padrão = 0.01
  Precision: Média = 0.67, Desvio Padrão = 0.01

Melhor F1-score (GridSearchCV no dataset completo): 0.58
Melhores parâmetros (GridSearchCV no dataset completo): {'nb__var_smoothing': 1e-09, 'pca__n_components': 10}

Métricas do Modelo Final Ajustado (no dataset completo):
  Acurácia: 0.82
  F1-Score: 0.58
  Recall: 0.50
  Precisão: 0.67


### Versão 4: Com Redução de Dimensionalidade (PCA) e Balanceamento de Classes

(Undersampling) Otimizado para F1-Score

Esta versão estende a anterior, incorporando uma etapa de balanceamento de classes utilizando `RandomUnderSampler` *dentro* do pipeline e dos loops de validação cruzada. O `undersampling` será aplicado apenas aos dados de treino/validação de cada fold, garantindo que o conjunto de teste permaneça com a distribuição de classes original para uma avaliação imparcial.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.decomposition import PCA
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, recall_score, precision_score

# 1. Defina o Pipeline com Undersampling
# Usamos ImbPipeline para que o undersampler seja aplicado apenas nos dados de treino de cada fold
pipe_pca_undersample = ImbPipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('undersampler', RandomUnderSampler(random_state=42)), # Undersampling para balancear as classes no treino
    ('nb', GaussianNB())
])

# 2. Defina o espaço de busca
param_grid_pca_undersample = {
    'pca__n_components': [2, 5, 10],
    'nb__var_smoothing': [1e-9, 1e-8, 1e-7]
}

# 3. Defina as métricas de scoring para cross_validate
scoring_metrics_undersample = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score)
}

# 4. Loop Interno: Ajuste fino (Tuning) com a métrica F1
inner_cv_pca_undersample = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
# O GridSearchCV otimiza baseado no F1-score
clf_pca_undersample = GridSearchCV(pipe_pca_undersample, param_grid=param_grid_pca_undersample,
                                   cv=inner_cv_pca_undersample, n_jobs=-1, scoring='f1')

# 5. Loop Externo: Avaliação Real (sem vazamento) com múltiplas métricas
outer_cv_pca_undersample = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
nested_scores_pca_undersample = cross_validate(clf_pca_undersample, X, y, cv=outer_cv_pca_undersample,
                                                 n_jobs=-1, scoring=scoring_metrics_undersample, return_estimator=True)

print(f"--- Versão: Com Redução de Dimensionalidade (PCA), Undersampling e Naive Bayes Otimizado para F1-Score ---")

print("\nMétricas da Validação Aninhada (Nested Cross-Validation):")
for metric_name, scores in nested_scores_pca_undersample.items():
    if metric_name.startswith('test_'): # Filtra apenas as pontuações de teste
        print(f"  {metric_name[5:].replace('_', ' ').capitalize()}: Média = {scores.mean():.2f}, Desvio Padrão = {scores.std():.2f}")

# Para obter o melhor resultado (parâmetros e score) do GridSearchCV após o Nested CV,
# precisamos ajustar o GridSearchCV uma vez no dataset completo.
clf_pca_undersample.fit(X, y)
print(f"\nMelhor F1-score (GridSearchCV no dataset completo): {clf_pca_undersample.best_score_:.2f}")
print(f"Melhores parâmetros (GridSearchCV no dataset completo): {clf_pca_undersample.best_params_}")

# Métricas do modelo final ajustado no dataset completo
y_pred_undersample = clf_pca_undersample.predict(X)
print("\nMétricas do Modelo Final Ajustado (no dataset completo):")
print(f"  Acurácia: {accuracy_score(y, y_pred_undersample):.2f}")
print(f"  F1-Score: {f1_score(y, y_pred_undersample):.2f}")
print(f"  Recall: {recall_score(y, y_pred_undersample):.2f}")
print(f"  Precisão: {precision_score(y, y_pred_undersample):.2f}")

--- Versão: Com Redução de Dimensionalidade (PCA), Undersampling e Naive Bayes Otimizado para F1-Score ---

Métricas da Validação Aninhada (Nested Cross-Validation):
  Accuracy: Média = 0.76, Desvio Padrão = 0.01
  F1: Média = 0.63, Desvio Padrão = 0.01
  Recall: Média = 0.80, Desvio Padrão = 0.01
  Precision: Média = 0.52, Desvio Padrão = 0.01

Melhor F1-score (GridSearchCV no dataset completo): 0.63
Melhores parâmetros (GridSearchCV no dataset completo): {'nb__var_smoothing': 1e-09, 'pca__n_components': 10}

Métricas do Modelo Final Ajustado (no dataset completo):
  Acurácia: 0.76
  F1-Score: 0.63
  Recall: 0.80
  Precisão: 0.51


## Características selecionadas

### Versão 6: Com Seleção de Características (`SelectKBest`) Sem Balanceamento de Classes, Otimizado para F1-Score

Esta versão utiliza `SelectKBest` para a seleção de características sem aplicar balanceamento de classes, com o `GridSearchCV` otimizando o modelo Gaussian Naive Bayes para o F1-score, e avaliando múltiplas métricas através de validação cruzada aninhada com 10 folds estratificados.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, recall_score, precision_score

# 1. Defina o Pipeline
pipe_selector_f1 = Pipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('nb', GaussianNB())
])

# 2. Defina o espaço de busca
param_grid_selector_f1 = {
    'selector__k': [5, 10, 20], # Tente diferentes números de características
    'nb__var_smoothing': [1e-9, 1e-8, 1e-7] # Parâmetro para Gaussian Naive Bayes
}

# 3. Defina as métricas de scoring para cross_validate
scoring_metrics_selector_f1 = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score)
}

# 4. Loop Interno: Ajuste fino (Tuning) com a métrica F1
inner_cv_selector_f1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
clf_selector_f1 = GridSearchCV(pipe_selector_f1, param_grid=param_grid_selector_f1,
                               cv=inner_cv_selector_f1, n_jobs=-1, scoring='f1')

# 5. Loop Externo: Avaliação Real (sem vazamento) com múltiplas métricas
outer_cv_selector_f1 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
nested_scores_selector_f1 = cross_validate(clf_selector_f1, X, y, cv=outer_cv_selector_f1,
                                                 n_jobs=-1, scoring=scoring_metrics_selector_f1, return_estimator=True)

print(f"--- Versão: Com Seleção de Características (SelectKBest e Naive Bayes) Otimizado para F1-Score ---")

print("\nMétricas da Validação Aninhada (Nested Cross-Validation):")
for metric_name, scores in nested_scores_selector_f1.items():
    if metric_name.startswith('test_'): # Filtra apenas as pontuações de teste
        print(f"  {metric_name[5:].replace('_', ' ').capitalize()}: Média = {scores.mean():.2f}, Desvio Padrão = {scores.std():.2f}")

# Para obter o melhor resultado (parâmetros e score) do GridSearchCV após o Nested CV,
# precisamos ajustar o GridSearchCV uma vez no dataset completo.
clf_selector_f1.fit(X, y)
print(f"\nMelhor F1-score (GridSearchCV no dataset completo): {clf_selector_f1.best_score_:.2f}")
print(f"Melhores parâmetros (GridSearchCV no dataset completo): {clf_selector_f1.best_params_}")

# Métricas do modelo final ajustado no dataset completo
y_pred_selector_f1 = clf_selector_f1.predict(X)
print("\nMétricas do Modelo Final Ajustado (no dataset completo):")
print(f"  Acurácia: {accuracy_score(y, y_pred_selector_f1):.2f}")
print(f"  F1-Score: {f1_score(y, y_pred_selector_f1):.2f}")
print(f"  Recall: {recall_score(y, y_pred_selector_f1):.2f}")
print(f"  Precisão: {precision_score(y, y_pred_selector_f1):.2f}")

--- Versão: Com Seleção de Características (SelectKBest e Naive Bayes) Otimizado para F1-Score ---

Métricas da Validação Aninhada (Nested Cross-Validation):
  Accuracy: Média = 0.75, Desvio Padrão = 0.01
  F1: Média = 0.63, Desvio Padrão = 0.01
  Recall: Média = 0.84, Desvio Padrão = 0.01
  Precision: Média = 0.50, Desvio Padrão = 0.01

Melhor F1-score (GridSearchCV no dataset completo): 0.63
Melhores parâmetros (GridSearchCV no dataset completo): {'nb__var_smoothing': 1e-09, 'selector__k': 20}

Métricas do Modelo Final Ajustado (no dataset completo):
  Acurácia: 0.75
  F1-Score: 0.63
  Recall: 0.85
  Precisão: 0.50


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [1] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


### Versão 7: Com Seleção de Características (`SelectKBest`) e Balanceamento de Classes (Undersampling), Otimizado para F1-Score

Esta versão combina a seleção de características com `SelectKBest` e o balanceamento de classes através de `RandomUnderSampler` dentro do pipeline. O `GridSearchCV` otimiza o modelo Gaussian Naive Bayes para F1-score em uma validação cruzada aninhada com 10 folds estratificados, e as métricas completas são avaliadas.

In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, f1_score, recall_score, precision_score

# 1. Defina o Pipeline com Undersampling
pipe_selector_undersample = ImbPipeline([
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(score_func=f_classif)),
    ('undersampler', RandomUnderSampler(random_state=42)), # Undersampling para balancear as classes no treino
    ('nb', GaussianNB())
])

# 2. Defina o espaço de busca
param_grid_selector_undersample = {
    'selector__k': [5, 10, 20],
    'nb__var_smoothing': [1e-9, 1e-8, 1e-7]
}

# 3. Defina as métricas de scoring para cross_validate
scoring_metrics_selector_undersample = {
    'accuracy': make_scorer(accuracy_score),
    'f1': make_scorer(f1_score),
    'recall': make_scorer(recall_score),
    'precision': make_scorer(precision_score)
}

# 4. Loop Interno: Ajuste fino (Tuning) com a métrica F1
inner_cv_selector_undersample = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
clf_selector_undersample = GridSearchCV(pipe_selector_undersample, param_grid=param_grid_selector_undersample,
                                        cv=inner_cv_selector_undersample, n_jobs=-1, scoring='f1')

# 5. Loop Externo: Avaliação Real (sem vazamento) com múltiplas métricas
outer_cv_selector_undersample = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
nested_scores_selector_undersample = cross_validate(clf_selector_undersample, X, y, cv=outer_cv_selector_undersample,
                                                      n_jobs=-1, scoring=scoring_metrics_selector_undersample, return_estimator=True)

print(f"--- Versão: Com Seleção de Características (SelectKBest), Undersampling e Naive Bayes Otimizado para F1-Score ---")

print("\nMétricas da Validação Aninhada (Nested Cross-Validation):")
for metric_name, scores in nested_scores_selector_undersample.items():
    if metric_name.startswith('test_'): # Filtra apenas as pontuações de teste
        print(f"  {metric_name[5:].replace('_', ' ').capitalize()}: Média = {scores.mean():.2f}, Desvio Padrão = {scores.std():.2f}")

# Para obter o melhor resultado (parâmetros e score) do GridSearchCV após o Nested CV,
# precisamos ajustar o GridSearchCV uma vez no dataset completo.
clf_selector_undersample.fit(X, y)
print(f"\nMelhor F1-score (GridSearchCV no dataset completo): {clf_selector_undersample.best_score_:.2f}")
print(f"Melhores parâmetros (GridSearchCV no dataset completo): {clf_selector_undersample.best_params_}")

# Métricas do modelo final ajustado no dataset completo
y_pred_selector_undersample = clf_selector_undersample.predict(X)
print("\nMétricas do Modelo Final Ajustado (no dataset completo):")
print(f"  Acurácia: {accuracy_score(y, y_pred_selector_undersample):.2f}")
print(f"  F1-Score: {f1_score(y, y_pred_selector_undersample):.2f}")
print(f"  Recall: {recall_score(y, y_pred_selector_undersample):.2f}")
print(f"  Precisão: {precision_score(y, y_pred_selector_undersample):.2f}")

--- Versão: Com Seleção de Características (SelectKBest), Undersampling e Naive Bayes Otimizado para F1-Score ---

Métricas da Validação Aninhada (Nested Cross-Validation):
  Accuracy: Média = 0.75, Desvio Padrão = 0.01
  F1: Média = 0.63, Desvio Padrão = 0.01
  Recall: Média = 0.85, Desvio Padrão = 0.01
  Precision: Média = 0.50, Desvio Padrão = 0.01

Melhor F1-score (GridSearchCV no dataset completo): 0.63
Melhores parâmetros (GridSearchCV no dataset completo): {'nb__var_smoothing': 1e-09, 'selector__k': 20}

Métricas do Modelo Final Ajustado (no dataset completo):
  Acurácia: 0.75
  F1-Score: 0.63
  Recall: 0.85
  Precisão: 0.50


/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:111: UserWarning: Features [1] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
/usr/local/lib/python3.12/dist-packages/sklearn/feature_selection/_univariate_selection.py:112: RuntimeWarning: invalid value encountered in divide
  f = msb / msw
